In [ ]:
"""
CIFAR-10 통합 실습 스크립트 — ver1
=====================================================================
ver1 변경 사항 (대비 v0):
  [A] CustomCNN → CifarResNet (He et al. 2016 §4.2 small-input ResNet, ResNet-20/32/56)
      - 3x3 stride-1 stem, NO MaxPool, 16→32→64 채널, 깊이 = 6n+2
      - From-scratch 학습 (32×32 native 해상도 보존)
  [B] ResNet50 frozen-fc → Deep two-stage fine-tuning
      - Stage 1: head warmup (backbone frozen)
      - Stage 2: full unfreeze, discriminative LR (backbone << head)
  [공통] Best-model checkpoint, CosineAnnealingLR, CIFAR-10 표준 augmentation
=====================================================================
"""

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#해당 버전의 한계점
*   CIFAR ResNet from-scratch는 **epoch 수에 민감**: paper의 91.25%는 ~200 epochs 학습 결과. 본 스크립트는 60 epochs를 기본값으로 두어 약 88–90% 범위가 현실적.
*   **Stage 2 LR 선택의 휴리스틱성**: backbone LR 1e-5는 일반적 값이나, 도메인 차이가 크면 1e-4 ~ 5e-6 범위에서 grid search 필요.
*   **Weight decay와 BN의 상호작용**: 본 구현은 모든 파라미터에 동일한 WD 적용. 엄밀히는 BN affine과 bias는 WD 제외가 권장되나, 단순성을 위해 통일.
*   **No mixup/cutmix**: 추가 정규화 기법은 의도적으로 제외. 본 ver1의 목적은 두 방향(CIFAR-style 아키텍처, deep finetune)의 직접 효과 검증.
*   **단일 seed 실행**: 통계적 신뢰구간 산출에는 multi-seed 반복 필요.

In [1]:
from __future__ import annotations

In [2]:
import copy
import os
import random
from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple

In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.datasets import CIFAR10

In [4]:
# =====================================================================
# 1. Reproducibility
# =====================================================================
def set_seed(seed: int = 2023) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

In [5]:
# =====================================================================
# 2. Configuration
# =====================================================================
@dataclass
class Config:
    seed: int = 2023
    data_root: str = "./data"
    val_ratio: float = 0.2
    num_classes: int = 10

    # CifarResNet
    cifar_input_size: int = 32
    cifar_resnet_depth: int = 20          # 20 / 32 / 56 / 110 (must satisfy 6n+2)
    epochs_cifar: int = 60
    lr_cifar: float = 0.1                 # SGD initial LR (paper default)
    momentum: float = 0.9
    weight_decay_cifar: float = 1e-4

    # ResNet50 transfer (two-stage fine-tune)
    pretrained_input_size: int = 224
    epochs_warmup: int = 3                # Stage 1: head only
    epochs_finetune: int = 10             # Stage 2: full unfreeze
    lr_head_warmup: float = 1e-3          # Stage 1 head LR
    lr_backbone_ft: float = 1e-5          # Stage 2 backbone LR (very small)
    lr_head_ft: float = 1e-4              # Stage 2 head LR (smaller than warmup)
    weight_decay_ft: float = 1e-4

    # Loader
    batch_size: int = 128
    num_workers: int = 2

    device: str = "cuda" if torch.cuda.is_available() else "cpu"

*   He et al. (2016, Deep Residual Learning for Image Recognition) Section 4.2의 설계를 그대로 따른다. 깊이 = 6n+2 → ResNet-20 (n=3), ResNet-32 (n=5), ResNet-56 (n=9), ResNet-110 (n=18).
*   논문에선s ~200 epochs로 91.25% 달성, 그러나 이 스크립트에서는 일단 60 epochs로 설정함

In [6]:
# =====================================================================
# 3. Data loading
# =====================================================================
def load_cifar10_splits(cfg: Config) -> Tuple[
    Tuple[np.ndarray, np.ndarray],
    Tuple[np.ndarray, np.ndarray],
    Tuple[np.ndarray, np.ndarray],
]:
    train_full = CIFAR10(root=cfg.data_root, train=True, download=True)
    test_set = CIFAR10(root=cfg.data_root, train=False, download=True)

    x_full = train_full.data
    y_full = np.array(train_full.targets, dtype=np.int64)
    x_test = test_set.data
    y_test = np.array(test_set.targets, dtype=np.int64)

    x_train, x_val, y_train, y_val = train_test_split(
        x_full, y_full,
        test_size=cfg.val_ratio,
        random_state=cfg.seed,
        stratify=y_full,
    )
    return (x_train, y_train), (x_val, y_val), (x_test, y_test)

In [7]:
# =====================================================================
# 4. Dataset
# =====================================================================
class CIFAR10Dataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, transform: Optional[Callable] = None):
        assert len(images) == len(labels), "images/labels length mismatch"
        self.images = images
        self.labels = labels.astype(np.int64)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int):
        img = Image.fromarray(self.images[idx])
        label = int(self.labels[idx])
        if self.transform is not None:
            img = self.transform(img)
        return img, label

In [8]:
# =====================================================================
# 5. Transforms
# =====================================================================
CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
CIFAR_STD = [0.2470, 0.2435, 0.2616]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [9]:
def build_transforms(
    input_size: int,
    augment: bool,
    normalize: Optional[str] = None,    # None | "cifar" | "imagenet"
) -> transforms.Compose:
    """
    augment=True: train 전용 (RandomCrop padding=4 + HFlip; CIFAR 표준).
    normalize="cifar" : CIFAR-10 train set mean/std (from-scratch 학습용)
    normalize="imagenet": ImageNet mean/std (pretrained ResNet 경로)
    """
    ops: List[Callable] = [transforms.Resize((input_size, input_size))]
    if augment:
        ops += [
            transforms.RandomCrop(input_size, padding=4),
            transforms.RandomHorizontalFlip(p=0.5),
        ]
    ops.append(transforms.ToTensor())
    if normalize == "cifar":
        ops.append(transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD))
    elif normalize == "imagenet":
        ops.append(transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD))
    return transforms.Compose(ops)

In [10]:
# =====================================================================
# 6. DataLoader builder
# =====================================================================
def build_loaders(
    splits: Tuple[Tuple[np.ndarray, np.ndarray], ...],
    cfg: Config,
    input_size: int,
    normalize: Optional[str],
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    (x_tr, y_tr), (x_va, y_va), (x_te, y_te) = splits

    train_tf = build_transforms(input_size, augment=True,  normalize=normalize)
    eval_tf  = build_transforms(input_size, augment=False, normalize=normalize)

    train_ds = CIFAR10Dataset(x_tr, y_tr, transform=train_tf)
    val_ds   = CIFAR10Dataset(x_va, y_va, transform=eval_tf)
    test_ds  = CIFAR10Dataset(x_te, y_te, transform=eval_tf)

    pin = cfg.device == "cuda"
    common = dict(batch_size=cfg.batch_size, num_workers=cfg.num_workers, pin_memory=pin)
    return (
        DataLoader(train_ds, shuffle=True, drop_last=False, **common),
        DataLoader(val_ds,   shuffle=False, drop_last=False, **common),
        DataLoader(test_ds,  shuffle=False, drop_last=False, **common),
    )

In [11]:
# =====================================================================
# 7a. Model — CIFAR-style ResNet (He et al. 2016, §4.2)
# =====================================================================
class BasicBlock(nn.Module):
    """
    Residual block (no bottleneck).
    Conv3x3 → BN → ReLU → Conv3x3 → BN → (+ shortcut) → ReLU.
    Shortcut: identity if shapes match; else 1x1 stride-s conv (Option B).
    """
    expansion = 1

    def __init__(self, in_planes: int, planes: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        if stride != 1 or in_planes != planes:
            # Option B: projection shortcut (1x1 conv + BN)
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = F.relu(out, inplace=True)
        return out

In [12]:
class CifarResNet(nn.Module):
    """
    CIFAR-style ResNet. depth = 6n + 2.
        depth=20 → n=3, depth=32 → n=5, depth=56 → n=9, depth=110 → n=18.

    Input (B, 3, 32, 32) → logits (B, num_classes).

    Stem: 3x3 conv, 16 ch, stride 1 (NO 7x7, NO MaxPool).
    Stage 1: n blocks, 16 ch, stride 1   →  32×32
    Stage 2: n blocks, 32 ch, first stride 2 → 16×16
    Stage 3: n blocks, 64 ch, first stride 2 →  8×8
    Head:   AdaptiveAvgPool2d(1) → FC(64, num_classes).

    Optional zero-init of the second BN gamma in each block ("zero-gamma" trick)
    is included; it improves convergence by making each residual branch
    initially behave as identity (cf. He et al. and Bag of Tricks for CNNs,
    He et al. 2019).
    """

    def __init__(self, depth: int = 20, num_classes: int = 10, zero_init_residual: bool = True) -> None:
        super().__init__()
        assert (depth - 2) % 6 == 0, f"depth must satisfy 6n+2, got {depth}"
        n = (depth - 2) // 6

        self.in_planes = 16
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )
        self.stage1 = self._make_stage(16, n, stride=1)
        self.stage2 = self._make_stage(32, n, stride=2)
        self.stage3 = self._make_stage(64, n, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)

        self._init_weights(zero_init_residual=zero_init_residual)

    def _make_stage(self, planes: int, num_blocks: int, stride: int) -> nn.Sequential:
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, stride=s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def _init_weights(self, zero_init_residual: bool) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                nn.init.constant_(m.bias, 0.0)

        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.avgpool(x)
        x = x.flatten(start_dim=1)
        x = self.fc(x)
        return x

In [13]:
def cifar_resnet(depth: int = 20, num_classes: int = 10) -> CifarResNet:
    return CifarResNet(depth=depth, num_classes=num_classes)

In [14]:
# =====================================================================
# 7b. Model — ResNet50 transfer (with freeze/unfreeze utilities)
# =====================================================================
def build_resnet50_transfer(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    if freeze_backbone:
        for name, param in model.named_parameters():
            param.requires_grad = name.startswith("fc.")
    return model

In [15]:
def unfreeze_all(model: nn.Module) -> nn.Module:
    for p in model.parameters():
        p.requires_grad = True
    return model

In [16]:
def trainable_parameters(model: nn.Module) -> List[nn.Parameter]:
    return [p for p in model.parameters() if p.requires_grad]

In [17]:
def build_discriminative_optimizer(
    model: nn.Module,
    lr_backbone: float,
    lr_head: float,
    weight_decay: float,
    head_prefix: str = "fc.",
) -> optim.Optimizer:
    """
    head_prefix로 시작하는 파라미터는 lr_head, 그 외는 lr_backbone 사용.
    """
    backbone_params, head_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        (head_params if name.startswith(head_prefix) else backbone_params).append(param)

    if not head_params:
        raise ValueError(f"No parameters matched head_prefix='{head_prefix}'")

    return optim.AdamW(
        [
            {"params": backbone_params, "lr": lr_backbone},
            {"params": head_params,     "lr": lr_head},
        ],
        weight_decay=weight_decay,
    )

In [18]:
# =====================================================================
# 8. Train / eval / fit engine
# =====================================================================
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: str,
) -> Tuple[float, float]:
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=-1) == y).sum().item()
        total_samples += x.size(0)
    return total_loss / total_samples, total_correct / total_samples

In [19]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
) -> Tuple[float, float]:
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=-1) == y).sum().item()
        total_samples += x.size(0)
    return total_loss / total_samples, total_correct / total_samples

In [20]:
def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: str,
    epochs: int,
    tag: str = "",
    scheduler: Optional[object] = None,
    monitor: str = "val_loss",
) -> Dict:
    """
    Returns:
        {"history": dict, "best_state": OrderedDict, "best_metric": float, "best_epoch": int}
    """
    assert monitor in {"val_loss", "val_acc"}
    history: Dict[str, List[float]] = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc")}
    best_value = float("inf") if monitor == "val_loss" else -float("inf")
    best_state, best_epoch = None, -1

    def is_better(curr: float, best: float) -> bool:
        return curr < best if monitor == "val_loss" else curr > best

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[k].append(v)

        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(va_loss)
            else:
                scheduler.step()

        curr = va_loss if monitor == "val_loss" else va_acc
        if is_better(curr, best_value):
            best_value = curr
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch

        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"[{tag}] {epoch:02d}/{epochs} | lr {lr_now:.2e} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print(f"[{tag}] best {monitor} = {best_value:.4f} at epoch {best_epoch}")
    return {"history": history, "best_state": best_state, "best_metric": best_value, "best_epoch": best_epoch}

In [21]:
# =====================================================================
# 9. Two-stage fine-tune helper
# =====================================================================
def two_stage_finetune(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    cfg: Config,
    tag: str = "ResNet50",
) -> Dict:
    """
    Stage 1: backbone frozen, train head only (epochs_warmup).
    Stage 2: unfreeze all, discriminative LR (epochs_finetune).
    Best state는 stage 2의 best (overall best는 stage 2가 항상 더 우수하다고 가정).
    """
    # ---- Stage 1: head warmup ----
    print(f"\n--- [{tag}] Stage 1: head warmup (backbone frozen) ---")
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith("fc.")
    optim_s1 = optim.AdamW(
        trainable_parameters(model),
        lr=cfg.lr_head_warmup,
        weight_decay=cfg.weight_decay_ft,
    )
    res_s1 = fit(
        model, train_loader, val_loader, criterion, optim_s1, cfg.device,
        epochs=cfg.epochs_warmup, tag=f"{tag}-S1", scheduler=None, monitor="val_loss",
    )

    # Stage 1 best state로 복원 후 stage 2 시작
    model.load_state_dict(res_s1["best_state"])

    # ---- Stage 2: deep fine-tune ----
    print(f"\n--- [{tag}] Stage 2: full unfreeze with discriminative LR ---")
    unfreeze_all(model)
    optim_s2 = build_discriminative_optimizer(
        model,
        lr_backbone=cfg.lr_backbone_ft,
        lr_head=cfg.lr_head_ft,
        weight_decay=cfg.weight_decay_ft,
    )
    sched_s2 = optim.lr_scheduler.CosineAnnealingLR(optim_s2, T_max=cfg.epochs_finetune)
    res_s2 = fit(
        model, train_loader, val_loader, criterion, optim_s2, cfg.device,
        epochs=cfg.epochs_finetune, tag=f"{tag}-S2", scheduler=sched_s2, monitor="val_loss",
    )

    return {
        "stage1": res_s1,
        "stage2": res_s2,
        "history_concat": {
            k: res_s1["history"][k] + res_s2["history"][k] for k in res_s1["history"]
        },
        "final_best_state": res_s2["best_state"],
    }

In [22]:
# =====================================================================
# 10. Optional: history plot
# =====================================================================
def plot_histories(histories: Dict[str, Dict[str, List[float]]], save_path: Optional[str] = None) -> None:
    """English titles/labels (Korean font 미설정 환경 대응)."""
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for tag, h in histories.items():
        axes[0].plot(h["train_loss"], label=f"{tag} train")
        axes[0].plot(h["val_loss"],   label=f"{tag} val", linestyle="--")
        axes[1].plot(h["train_acc"],  label=f"{tag} train")
        axes[1].plot(h["val_acc"],    label=f"{tag} val", linestyle="--")
    for ax, ylab in zip(axes, ("Loss", "Accuracy")):
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylab)
        ax.set_title(ylab)
        ax.legend(fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()

In [23]:
# =====================================================================
# 11. Main
# =====================================================================
def main() -> Dict:
    cfg = Config()
    set_seed(cfg.seed)
    print(f"Device: {cfg.device} | CIFAR ResNet depth: {cfg.cifar_resnet_depth}")

    splits = load_cifar10_splits(cfg)
    criterion = nn.CrossEntropyLoss()

    # -----------------------------------------------------------------
    # (A) CIFAR-style ResNet (from scratch, 32×32 native)
    # -----------------------------------------------------------------
    print("\n=== A. CIFAR-style ResNet — from scratch, 32×32 native ===")
    train_a, val_a, test_a = build_loaders(
        splits, cfg, input_size=cfg.cifar_input_size, normalize="cifar"
    )
    model_a = cifar_resnet(depth=cfg.cifar_resnet_depth, num_classes=cfg.num_classes).to(cfg.device)

    n_params_a = sum(p.numel() for p in model_a.parameters())
    print(f"[CifarResNet-{cfg.cifar_resnet_depth}] params = {n_params_a:,}")

    optim_a = optim.SGD(
        model_a.parameters(),
        lr=cfg.lr_cifar,
        momentum=cfg.momentum,
        weight_decay=cfg.weight_decay_cifar,
        nesterov=True,
    )
    sched_a = optim.lr_scheduler.CosineAnnealingLR(optim_a, T_max=cfg.epochs_cifar)
    res_a = fit(
        model_a, train_a, val_a, criterion, optim_a, cfg.device,
        epochs=cfg.epochs_cifar, tag=f"CifarResNet-{cfg.cifar_resnet_depth}",
        scheduler=sched_a, monitor="val_loss",
    )

    model_a.load_state_dict(res_a["best_state"])
    test_loss_a, test_acc_a = evaluate(model_a, test_a, criterion, cfg.device)
    print(f"[CifarResNet-{cfg.cifar_resnet_depth}] BEST-MODEL TEST loss {test_loss_a:.4f} acc {test_acc_a:.4f}")

    # -----------------------------------------------------------------
    # (B) ResNet50 deep two-stage fine-tuning (224×224, ImageNet normalize)
    # -----------------------------------------------------------------
    print("\n=== B. ResNet50 — two-stage deep fine-tuning ===")
    train_b, val_b, test_b = build_loaders(
        splits, cfg, input_size=cfg.pretrained_input_size, normalize="imagenet"
    )
    model_b = build_resnet50_transfer(num_classes=cfg.num_classes, freeze_backbone=True).to(cfg.device)
    n_params_b = sum(p.numel() for p in model_b.parameters())
    print(f"[ResNet50] params = {n_params_b:,}")

    res_b = two_stage_finetune(model_b, train_b, val_b, criterion, cfg, tag="ResNet50")
    model_b.load_state_dict(res_b["final_best_state"])
    test_loss_b, test_acc_b = evaluate(model_b, test_b, criterion, cfg.device)
    print(f"[ResNet50] BEST-MODEL TEST loss {test_loss_b:.4f} acc {test_acc_b:.4f}")

    # -----------------------------------------------------------------
    # Summary
    # -----------------------------------------------------------------
    print("\n=== Summary ===")
    print(f"  CifarResNet-{cfg.cifar_resnet_depth} (from scratch) : test acc = {test_acc_a:.4f}")
    print(f"  ResNet50 (deep fine-tune)           : test acc = {test_acc_b:.4f}")

    # 시각화 (선택)
    # plot_histories({
    #     f"CifarResNet-{cfg.cifar_resnet_depth}": res_a["history"],
    #     "ResNet50": res_b["history_concat"],
    # }, save_path="curves_ver1.png")

    return {
        "cifar_resnet": {
            "depth": cfg.cifar_resnet_depth,
            "history": res_a["history"],
            "best_epoch": res_a["best_epoch"],
            "test_loss": test_loss_a,
            "test_acc": test_acc_a,
        },
        "resnet50_ft": {
            "history_concat": res_b["history_concat"],
            "stage1_best_epoch": res_b["stage1"]["best_epoch"],
            "stage2_best_epoch": res_b["stage2"]["best_epoch"],
            "test_loss": test_loss_b,
            "test_acc": test_acc_b,
        },
    }

In [24]:
if __name__ == "__main__":
    main()

Device: cuda | CIFAR ResNet depth: 20


100%|██████████| 170M/170M [00:06<00:00, 27.4MB/s]



=== A. CIFAR-style ResNet — from scratch, 32×32 native ===
[CifarResNet-20] params = 272,474
[CifarResNet-20] 01/60 | lr 9.99e-02 | train loss 1.6166 acc 0.4018 | val loss 1.4006 acc 0.4936
[CifarResNet-20] 02/60 | lr 9.97e-02 | train loss 1.1297 acc 0.5964 | val loss 1.2363 acc 0.5853
[CifarResNet-20] 03/60 | lr 9.94e-02 | train loss 0.9066 acc 0.6774 | val loss 1.0155 acc 0.6710
[CifarResNet-20] 04/60 | lr 9.89e-02 | train loss 0.7675 acc 0.7306 | val loss 0.8342 acc 0.7207
[CifarResNet-20] 05/60 | lr 9.83e-02 | train loss 0.6780 acc 0.7644 | val loss 0.8987 acc 0.7122
[CifarResNet-20] 06/60 | lr 9.76e-02 | train loss 0.6219 acc 0.7853 | val loss 0.7746 acc 0.7512
[CifarResNet-20] 07/60 | lr 9.67e-02 | train loss 0.5774 acc 0.7986 | val loss 0.6756 acc 0.7715
[CifarResNet-20] 08/60 | lr 9.57e-02 | train loss 0.5399 acc 0.8117 | val loss 0.6384 acc 0.7911
[CifarResNet-20] 09/60 | lr 9.46e-02 | train loss 0.5097 acc 0.8236 | val loss 0.7562 acc 0.7592
[CifarResNet-20] 10/60 | lr 9.33e

100%|██████████| 97.8M/97.8M [00:00<00:00, 208MB/s]


[ResNet50] params = 23,528,522

--- [ResNet50] Stage 1: head warmup (backbone frozen) ---
[ResNet50-S1] 01/3 | lr 1.00e-03 | train loss 0.8999 acc 0.7423 | val loss 0.6273 acc 0.7982
[ResNet50-S1] 02/3 | lr 1.00e-03 | train loss 0.5769 acc 0.8146 | val loss 0.5422 acc 0.8214
[ResNet50-S1] 03/3 | lr 1.00e-03 | train loss 0.5119 acc 0.8328 | val loss 0.5142 acc 0.8269
[ResNet50-S1] best val_loss = 0.5142 at epoch 3

--- [ResNet50] Stage 2: full unfreeze with discriminative LR ---
[ResNet50-S2] 01/10 | lr 9.76e-06 | train loss 0.3506 acc 0.8823 | val loss 0.2820 acc 0.9055
[ResNet50-S2] 02/10 | lr 9.05e-06 | train loss 0.2196 acc 0.9253 | val loss 0.2168 acc 0.9279
[ResNet50-S2] 03/10 | lr 7.94e-06 | train loss 0.1672 acc 0.9426 | val loss 0.1863 acc 0.9397
[ResNet50-S2] 04/10 | lr 6.55e-06 | train loss 0.1305 acc 0.9564 | val loss 0.1707 acc 0.9451
[ResNet50-S2] 05/10 | lr 5.00e-06 | train loss 0.1062 acc 0.9636 | val loss 0.1637 acc 0.9456
[ResNet50-S2] 06/10 | lr 3.45e-06 | train loss 

#1. ver0 -> ver1 정량 비교
*   From-scratch (CustomCNN 2-block -> CifarResNet-20): **0.7155 (ver0) -> 0.8902 (ver1); +0.1747**
*   Pretrained (ResNet50 frozen -> deep finetune): **0.8480 (ver0) -> 0.9527 (ver1); +0.1047**



---


#2. CifarResNet-20 진단
관찰 1 — **Cosine annealing의 효과가 명확**
*   Epoch 1–20: val 곡선이 매우 noisy (epoch 9 0.7592 → 11 0.8069 → 12 0.7854 등 ±0.02 진동)
*   Epoch 35 이후: LR이 0.04 이하로 감소하며 val 곡선이 안정화
*   최저 val_loss = 0.3834 at epoch 54 → **거의 종점 부근에서 best 달성. 이는 CosineLR schedule이 끝까지 의미 있게 작동했음을 의미**

<br>

관찰 2 — **명확한 overfitting 신호**
*   Final epoch 60: train_acc 0.9844 vs val_acc 0.9020 → gap 8.24%p
*   Train loss 0.0503 vs val loss 0.3851 → 약 7.6배 차이
*   ver0 CustomCNN의 gap +0.92%p와 정반대 양상: **ver1의 ResNet은 capacity가 충분하여 overfit이 가능해진 것**

<br>

관찰 3 — **Best epoch와 final epoch의 일관성**
*   Epoch 54 (best val_loss): val_acc 0.9013
*   Epoch 60 (final): val_acc 0.9020
*   두 시점 가중치 차이가 매우 작음 → Cosine LR이 후반에 사실상 학습을 멈추기 때문. 이 경우 best-model checkpoint의 실질 효과는 제한적

<br>

관찰 4 — Val-Test 간극 (1.11%p)
*   Best val_acc 0.9033 vs Test 0.8902 → 1.31%p 하락
*   ver0의 0.34%p 대비 큼. 모델이 val에 일정 정도 적응한 신호 가능성 (val_loss로 best selection을 했으므로)



---


#3. ResNet50 Deep Fine-tune 진단

관찰 5 — Stage 1 → **Stage 2 전환 효과**
*   Stage 1 종료 (epoch 3): val_acc 0.8269
*   SStage 2 epoch 1: val_acc 0.9055 → 단일 epoch에 +7.86%p
*   S이는 backbone unfreeze + discriminative LR가 **frozen-fc 단계에서 추출 불가능했던 representation을 즉시 활용**함을 보여줌

<br>

관찰 6 — **학습이 아직 완료되지 않음**
*   Final val_loss 0.1540 (epoch 10), 그 직전 0.1541, 0.1545, 0.1554 → 미세하지만 단조 감소 지속
*   LR이 0에 도달했으므로 더 학습하려면 **schedule 재설계 필요** (예: T_max=15로 늘리거나 두 번째 cosine cycle 추가)

<br>

관찰 7 — **Generalization gap이 매우 작음**
*   Final train_acc 0.9769 vs val_acc 0.9512 → gap 2.57%p
*   CifarResNet의 8.24%p와 대조적. **ImageNet pretrained feature가 강한 regularizer 역할**

<br>

관찰 8 — **Val < Test 역전**
*   Final val_acc 0.9512 vs Test 0.9527 → test가 0.15%p 더 높음
*   ver0와 동일한 패턴. **data split 무작위성 + 정규화의 효과**



---


#4. 4개 모델 결과 시각화

In [2]:
from IPython.display import HTML

# 1. html 파일이 있는 구글 드라이브 경로를 정확히 입력하세요.
html_path = '/content/drive/MyDrive/Colab Notebooks/PyTorch 실습 /cifar10_v0_v1_comparison.html'

# 2. 파일을 파이썬으로 열어서 내용을 읽습니다.
with open(html_path, 'r', encoding='utf-8') as f:
    html_content = f.read()

# 3. 셀 아래에 출력합니다.
display(HTML(html_content))



---


#5. 파라미터 효율성 비교

*   CifarResNet-20
    *   Test acc: 0.8902
    *   Params: 272,474
    *   비고: from scratch, 60 ep

<br>

*   ResNet50 deep FT
    *   Test acc: 0.9527
    *   Params: 23,528,522
    *   비고: pretrained, 13 ep

<br>

*   ResNet50은 86배 많은 파라미터로 6.25%p 우위. 파라미터 1M당 정확도 환산시
      *   CifarResNet-20: 3.27 acc/M
      *   ResNet50: 0.041 acc/M
      *   CifarResNet이 파라미터 효율성에서 ~70배 우수



---


#6. 두 모델 결과 요약
관찰 9 — 학습 양상이 근본적으로 다름
*   CifarResNet: **overfit-prone** (train 98% / val 90%), pure visual learning
*   ResNet50 FT: **잘 정규화됨** (train 98% / val 95%), **feature adaptation 중심**

<br>

관찰 10 - 각 모델별 추가 개선 여지
*   CifarResNet 추가 개선 여지:
    *   **MixUp / CutMix** (더 강한 정규화): train-val gap 8%p를 줄이는 가장 직접적 방법. 예상 효과 +1–2%p
    *   **Depth 증가**: ResNet-32 (n=5, ~470k params) → ~91%, ResNet-56 (n=9, ~860k) → ~92–93% 추정
    *   **Epochs 증가**: 60 → 200 (paper baseline). Cosine schedule 늘리면 +0.5–1%p 추정
    *   Label smoothing 0.1: 가벼운 정규화, +0.3–0.5%p
*   ResNet50 추가 개선 여지:
    *   **Stage 2 epoch 증가**: 10 → 15 → 20. val_loss 단조 감소 중이므로 직접 효과 기대. epoch 추가만으로 95.5%+ 도달 가능
    *   **Weight decay 조정**: 현재 1e-4 → 5e-4로 시험
    *   **Test-time augmentation (TTA)**: HFlip 평균만 적용해도 +0.2–0.5%p



---


#7. 한계

*   본 결과는 단일 seed (2023)에서의 결과. **6.25%p의 두 모델 간 격차가 통계적으로 유의한지는 multi-seed로 확인 필요**.
*   Test set은 한 번만 평가되었으며, 이는 hyperparameter tuning의 결과로 best model을 선택했으므로 완벽한 unbiased estimate은 아님.
    *  **엄격하게는 hyperparameter는 val로, 최종 보고는 test로의 단일 평가가 원칙**이며, 본 실험은 그 원칙을 따랐음을 확인.